In [0]:
import time
from concurrent.futures import ThreadPoolExecutor
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType


MAX_RETRIES = 2

BASE_PATH = "/Users/yshsharma099@gmail.com/ecommerce_lakehouse_project (1)/Ecommerce-Databricks-Lakehouse/scripts/Silver"



In [0]:

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.monitoring")



In [0]:

pipeline_groups = [

    # Group 1
    [
        f"{BASE_PATH}/silver_customers",
        f"{BASE_PATH}/silver_sellers",
        f"{BASE_PATH}/silver_products",
        f"{BASE_PATH}/silver_category_translation",
        f"{BASE_PATH}/silver_geolocation"
    ],

    # Group 2
    [
        f"{BASE_PATH}/silver_orders"
    ],

    # Group 3
    [
        f"{BASE_PATH}/silver_order_items",
        f"{BASE_PATH}/silver_payments",
        f"{BASE_PATH}/silver_reviews"
    ]
]



In [0]:

log_schema = StructType([
    StructField("notebook", StringType(), True),
    StructField("status", StringType(), True),
    StructField("duration_seconds", DoubleType(), True),
    StructField("error_message", StringType(), True)
])

def log_execution(notebook, status, duration, error_msg=None):

    log_df = spark.createDataFrame(
        [(notebook, status, float(duration), error_msg)],
        schema=log_schema
    ).withColumn("execution_time", F.current_timestamp())

    (
        log_df.write
        .mode("append")
        .format("delta")
        .saveAsTable("workspace.monitoring.pipeline_logs")
    )




In [0]:

def run_notebook(nb):

    start_time = time.time()

    for attempt in range(MAX_RETRIES + 1):

        try:

            print(f"Running {nb} (Attempt {attempt+1})")

            dbutils.notebook.run(nb, timeout_seconds=0)

            duration = round(time.time() - start_time, 2)

            print(f"SUCCESS: {nb} completed in {duration}s")

            log_execution(nb, "SUCCESS", duration)

            return True

        except Exception as e:

            print(f"ERROR: {nb} failed attempt {attempt+1}")

            if attempt == MAX_RETRIES:

                duration = round(time.time() - start_time, 2)

                log_execution(nb, "FAILED", duration, str(e))

                raise e

            else:
                print(f"Retrying {nb}...")




In [0]:

pipeline_start = time.time()

print("====================================")
print("Starting Enterprise Silver Pipeline")
print("====================================")

for group_id, group in enumerate(pipeline_groups, start=1):

    print(f"\nExecuting Pipeline Group {group_id}")

    with ThreadPoolExecutor(max_workers=len(group)) as executor:
        results = list(executor.map(run_notebook, group))


pipeline_duration = round(time.time() - pipeline_start, 2)

print("\n====================================")
print("Silver Pipeline Completed")
print(f"Total Pipeline Time: {pipeline_duration} seconds")
print("====================================")